# Notebook 09: Job Title Quality Check (DE/EN) as a Decision-Making Aid

Profile expansion of KldB base future profiles using external text sources. Language issue: Much of the external job data is in English (global), but the target base is KldB or should be in German. Since there are not enough German data records available, English data must also be used. Therefore, here is the decision-making guide for the next steps in matching the future profiles and external data sets for the profile expansion of English data:
- Take a small sample of job titles (DE and EN)
- Check matching quality:
  1. DE job title -> KldB occupational designations (DE); DE is matched anyway, just as a test. Then:
  2. EN job titles -> ESCO Occupations (EN, would be optimal for profile expansion, but more effort and more complex)
  3. EN job titles -> KldB (EN proxy via `kldb_title_en`, automatically translated, would be a simpler approach)
  4. EN Job Title -> ISCO (EN, possibly as a bridge step, possibly better suited than ESCO?)
- Derive a trend from this to determine which matching method makes the most sense for profile expansion.

Outputs:
- Tables with best matches + scores
- Summary metrics
- Export of an Excel file

In [1]:
# Setup + Paths
from pathlib import Path
import pandas as pd
import numpy as np
import re
import json

# Project Root
PROJECT_ROOT = Path().resolve()
while not (PROJECT_ROOT / "data").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA = PROJECT_ROOT / "data"
DATA_INTERIM = DATA / "interim"
DATA_PROCESSED = DATA / "processed"
DATA_PROCESSED_EXTERNAL = DATA / "processed_external"
DATA_RAW = PROJECT_ROOT / "data" / "raw"

DOCS_PATH = DATA_PROCESSED_EXTERNAL / "documents_raw.parquet"
DOCS_LIGHT_PATH = DATA_PROCESSED_EXTERNAL / "documents_raw_light.parquet"

KDB_JOB_TITLES_LONG_PATH = DATA_PROCESSED / "kldb_job_titles_long.parquet"
KDB_MAPPING_PATH = DATA_INTERIM / "kldb_esco_mapping.parquet"
ESCO_OCC_PATH = DATA_INTERIM / "esco_occupations.parquet"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DOCS:", DOCS_LIGHT_PATH if DOCS_LIGHT_PATH.exists() else DOCS_PATH)

PROJECT_ROOT: C:\Users\sigle\OneDrive - Hochschule Reutlingen\Dokumente\Profilerweiterung
DOCS: C:\Users\sigle\OneDrive - Hochschule Reutlingen\Dokumente\Profilerweiterung\data\processed_external\documents_raw_light.parquet


## 1. Matching Approach

Fuzzy string matching as a baseline (trend test, based on Notebook 10b (see Karakatsianis et al., 2017; Senger et al., 2024; Zare et al. (2025)): `rapidfuzz`: fast & robust, otherwise fallback to `difflib` (small samples)).

Lightly normalize job titles:
- lowercase
- whitespace normalization
- remove typical additions (m/f/d, remote, etc.)

In [3]:
from rapidfuzz import process, fuzz

# normalization
def normalize_title(s: str) -> str:
    if s is None or (isinstance(s, float) and np.isnan(s)):
        return ""
    s = str(s).strip().lower()
    s = re.sub(r"\s+", " ", s)
    s = re.sub(r"[\u200b\u200c\u200d\ufeff]", "", s)  # zero
    s = re.sub(r"\((m/w/d|w/m/d|m/f/d)\)", "", s)
    s = re.sub(r"\b(remote|hybrid|full[- ]?time|part[- ]?time)\b", "", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

# Fuzzy Matching
try:
    from rapidfuzz import process, fuzz
    HAS_RAPIDFUZZ = True
except Exception:
    import difflib
    HAS_RAPIDFUZZ = False

def best_match(query: str, choices: list[str]):
    q = normalize_title(query)
    if not q or not choices:
        return None, 0.0
    
    if HAS_RAPIDFUZZ:
        m = process.extractOne(q, choices, scorer=fuzz.WRatio)
        if m is None:
            return None, 0.0
        best, score, _ = m
        return best, float(score)
    else:
        best = None
        best_score = 0.0
        for c in choices:
            s = difflib.SequenceMatcher(None, q, normalize_title(c)).ratio()
            if s > best_score:
                best_score = s
                best = c
        return best, float(best_score * 100)

print("rapidfuzz available:", HAS_RAPIDFUZZ)

rapidfuzz available: True


## 2. Preparing Files

### 2.1 Uploading Documents/Job Titles:

In [4]:
DOC_COLS = ["doc_id", "source_name", "source_type", "job_title_raw", "language"] # required columns

docs_path = DOCS_LIGHT_PATH if DOCS_LIGHT_PATH.exists() else DOCS_PATH
df_docs = pd.read_parquet(docs_path)

df_docs = df_docs[[c for c in DOC_COLS if c in df_docs.columns]].copy()
for c in ["source_name", "source_type", "job_title_raw", "language"]:
    if c in df_docs.columns:
        df_docs[c] = df_docs[c].fillna("").astype(str)

df_docs = df_docs[df_docs["job_title_raw"].str.strip() != ""].copy()

print("df_docs:", df_docs.shape)
df_docs.head(3)

df_docs: (192480, 5)


,doc_id,source_name,source_type,job_title_raw,language
0,921716,kaggle_linkedin_2023_2024_big,job_ad,Marketing Coordinator,en
1,1829192,kaggle_linkedin_2023_2024_big,job_ad,Mental Health Therapist/Counselor,en
2,10998357,kaggle_linkedin_2023_2024_big,job_ad,Assitant Restaurant Manager,en


### 2.2 Loading KLDB occupational titles (de):

In [5]:
df_kldb_titles = pd.read_parquet(KDB_JOB_TITLES_LONG_PATH)

candidates = ["job_title_de", "kldb_job_title_de", "berufsbenennung", "job_title"] # Column Selection
col_title = next((c for c in candidates if c in df_kldb_titles.columns), None)
if col_title is None:
    raise ValueError(f"Keine passende Jobtitel-Spalte gefunden. Vorhanden: {list(df_kldb_titles.columns)}")

kldb_choices_de = (
    df_kldb_titles[col_title]
    .astype("string")
    .str.strip()
    .dropna()
    .loc[lambda s: s.ne("")]
    .drop_duplicates()
    .tolist()
)

print("KldB DE choices:", len(kldb_choices_de))
kldb_choices_de[:10]

KldB DE choices: 18837


['3-D-Artist',
 '3-D-Designer/in',
 '3-D-Druck-Spezialist/in',
 'Abbrucharbeiter/in',
 'Abdichter/in (Dachdeckerei)',
 'Abdichtungspolier/in (Bauwerks- und Asphaltabdichtung)',
 'Abfallbeauftragte/r',
 'Abfallberater/in',
 'Abfallbeseitiger/in',
 'Abfalltechniker/in']

### 2.3 Loading ESCO Occupation Labels (en): Preferred vs. AltLabels as a List

In [6]:
# Load ESCO Occupation Labels: Preferred + AltLabels -> a choice list
df_esco_occ = pd.read_parquet(ESCO_OCC_PATH)

# Find Columns
pref_col = next(
    (c for c in ["pref_label_en", "preferredLabel_en", "preferredLabel", "occupation_title_en", "preferred_label"]
     if c in df_esco_occ.columns),
    None
)
alt_col = next(
    (c for c in ["alt_labels_en", "altLabels_en", "altLabels", "occupation_alt_labels_en", "alt_labels"]
     if c in df_esco_occ.columns),
    None
)

if pref_col is None:
    raise ValueError(f"PreferredLabel-Spalte nicht gefunden. Spalten: {list(df_esco_occ.columns)}")

import re
def split_altlabels(x):
    if x is None or (isinstance(x, float) and pd.isna(x)):
        return []
    s = str(x).strip()
    if not s:
        return []
    return [p.strip() for p in re.split(r"[;,]\s*", s) if p.strip()] # ESCO altLabels with ; or , separated

# Preferred Labels
esco_labels = (
    df_esco_occ[pref_col]
    .astype("string")
    .str.strip()
    .dropna()
    .loc[lambda s: s != ""]
    .tolist()
)

# with AltLabels
if alt_col is not None:
    for v in df_esco_occ[alt_col].dropna().tolist():
        esco_labels.extend(split_altlabels(v))

# dedupe
esco_choices_en = pd.Series(esco_labels).dropna()
esco_choices_en = esco_choices_en.astype(str).str.strip()
esco_choices_en = esco_choices_en[esco_choices_en != ""].drop_duplicates().tolist()

print("ESCO choices (preferred+alt):", len(esco_choices_en))
esco_choices_en[:10]

ESCO choices (preferred+alt): 6558


['technical director',
 'metal drawing machine operator',
 'precision device inspector',
 'air traffic safety technician',
 'hospitality revenue manager',
 'medical laboratory assistant',
 'asphalt laboratory technician',
 'primary school teaching assistant',
 'physiotherapist',
 'performing arts theatre instructor']

- Preferred Label: the unique main term for an ESCO concept
- Alternative Labels (Non-Preferred Terms): synonyms, alternative spellings, abbreviations, ...

### 2.4 Translating KLDB

- KldB EN Proxy (Translation of German occupational titles): Match English job titles with KldB occupational titles translated into English.
- Translation is expensive/unstable; save it once as a cache and then as a CSV/Parquet file.
- Automatic translation is computationally intensive, so the result file is saved, and there is no need to regenerate it when the process is run again.

In [7]:
from pathlib import Path
import random

TRANSLATION_CACHE_PATH = DATA_PROCESSED_EXTERNAL / "kldb_titles_de_to_en_cache.csv"

random.seed(42)
MAX_KLDB_TRANSLATE = None   # None for all
kldb_de_for_translation = kldb_choices_de if MAX_KLDB_TRANSLATE is None else random.sample(kldb_choices_de, min(MAX_KLDB_TRANSLATE, len(kldb_choices_de)))

print("Translate KldB titles:", len(kldb_de_for_translation))

Translate KldB titles: 18837


Translate:

In [ ]:
# pip install deep-translator # translator Lib, install once

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: C:\Users\sigle\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip
ERROR: Invalid requirement: '#': Expected package name at the start of dependency specifier
    #
    ^


In [10]:
# Local Translators Lib
from deep_translator import GoogleTranslator

def translate_de_to_en(text: str) -> str:
    text = str(text).strip()
    if not text:
        return ""
    try:
        return GoogleTranslator(source="de", target="en").translate(text)
    except Exception:
        return ""

Translation with cache: robust

A one-time translation is required because it is computationally intensive. For subsequent translations, the cached translation file can be used.

In [11]:
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, TimeoutError

AUTOSAVE_EVERY = 500 # to secure
TIMEOUT_S = 60  # if a translation takes longer -> skip it to avoid errors

# load Cache
if Path(TRANSLATION_CACHE_PATH).exists():
    df_cache = pd.read_csv(TRANSLATION_CACHE_PATH)
    cache = dict(zip(df_cache["de"], df_cache["en"]))
else:
    df_cache = pd.DataFrame(columns=["de", "en"])
    cache = {}

def translate_with_timeout(text: str, timeout_s: int = TIMEOUT_S) -> str:
    with ThreadPoolExecutor(max_workers=1) as ex: # Timeout over Thread
        fut = ex.submit(translate_de_to_en, text)
        try:
            return fut.result(timeout=timeout_s)
        except TimeoutError:
            return ""   # skip on Timeout
        except Exception:
            return ""   # skip if any other error occurs

new_rows = []
skipped = 0

for de_title in tqdm(kldb_de_for_translation, desc="Translating KldB titles"): # Progresbar
    if de_title in cache:
        continue

    en_title = translate_with_timeout(de_title)

    if en_title == "":
        skipped += 1
        print(f"SKIP (timeout/error): {de_title}")

    cache[de_title] = en_title
    new_rows.append({"de": de_title, "en": en_title})

    # Autosave
    if len(new_rows) >= AUTOSAVE_EVERY:
        df_new = pd.DataFrame(new_rows)
        df_cache = pd.concat([df_cache, df_new], ignore_index=True)
        df_cache.to_csv(TRANSLATION_CACHE_PATH, index=False, encoding="utf-8")
        new_rows = []

# Final save
if new_rows:
    df_new = pd.DataFrame(new_rows)
    df_cache = pd.concat([df_cache, df_new], ignore_index=True)
    df_cache.to_csv(TRANSLATION_CACHE_PATH, index=False, encoding="utf-8")
    print("Updated cache:", TRANSLATION_CACHE_PATH, "new:", len(df_new))
else:
    print("No new translations needed.")

# English choices for KldB (translated)
kldb_choices_en_translated = (
    pd.Series([cache.get(x, "") for x in kldb_de_for_translation])
    .dropna().astype(str).str.strip()
)

kldb_choices_en_translated = (
    kldb_choices_en_translated[kldb_choices_en_translated != ""]
    .drop_duplicates()
    .tolist()
)

print("KldB EN translated choices:", len(kldb_choices_en_translated))
print("New translations added to cache:", len(df_cache))
print("Skipped (timeout/error):", skipped)

kldb_choices_en_translated[:10]

Translating KldB titles: 100%|██████████| 18837/18837 [00:00<00:00, 3969259.20it/s]

No new translations needed.
KldB EN translated choices: 17742
New translations added to cache: 18837
Skipped (timeout/error): 0


['3D artist',
 '3D designer',
 '3D printing specialist',
 'Demolition worker',
 'Sealer (roofing)',
 'Sealing polisher (building and asphalt sealing)',
 'Waste representative',
 'Waste consultant',
 'Waste disposer',
 'Waste technician']

Result: Of 18,837 German KldB occupational titles, nearly all were translated. Thirty-five titles could not be translated due to timeouts or errors. After cleaning (removing empty values) and deduplication, 17,742 unique English proxy titles remain.

Speichern:

In [12]:
# Save Translated KldB Job Titles as a File
pd.DataFrame({"kldb_title_en": kldb_choices_en_translated}).to_csv(
    DATA_RAW / "kldb" / "kldb_titles_en.csv",
    index=False,
    encoding="utf-8"
)

In [13]:
# Also save as a parquet file (for faster reloading)
KDB_TITLES_EN_PROXY_PATH = DATA_PROCESSED_EXTERNAL / "kldb_titles_en_proxy.parquet"

df_kldb_en_proxy = pd.DataFrame({"kldb_title_de": list(cache.keys()),
                                 "kldb_title_en_proxy": list(cache.values())})

df_kldb_en_proxy.to_parquet(KDB_TITLES_EN_PROXY_PATH, index=False)
print("Saved:", KDB_TITLES_EN_PROXY_PATH)
print("kldb_choices_en_translated:", len(kldb_choices_en_translated))

Saved: C:\Users\sigle\OneDrive - Hochschule Reutlingen\Dokumente\Profilerweiterung\data\processed_external\kldb_titles_en_proxy.parquet
kldb_choices_en_translated: 17742


### 2.5 ISCO (EN): List of job titles from `ISCO08_Index.xlsx`

- `isco_title_en` from the KldB <-> ISCO mapping key consists of unit group titles (4 digits), which are more like broad categories rather than specific job titles.
- For realistic matching (job ad titles – ISCO), we therefore use `ISCO08_Index.xlsx` (source: ILOSTAT, https://ilostat.ilo.org/methods/concepts-and-definitions/classification-occupation/#elementor-toc__heading-anchor-4; December 27, 2025):
  - Contains English job titles (≈7k), but they are defined more narrowly and grouped more closely than in KldB or ESCO.
  - Each title is uniquely assigned to an ISCO-08 Unit Group (4-digit), making it possible to map back to KLDB/ESCO
- Sample of English job titles to be matched against this ISCO job title list for comparison with ESCO and the KldB-EN proxy.

load ISCO08_Index.xlsx, isco_choices_en_titles:

In [14]:
ISCO_INDEX_PATH = DATA_RAW / "isco" / "ISCO08_Index.xlsx"  # Path

df_isco_idx = pd.read_excel(ISCO_INDEX_PATH)

print("ISCO index columns:", list(df_isco_idx.columns))
df_isco_idx.head(3)

ISCO index columns: ['ISCO-08', 'ISCO-88', 'English title']


,ISCO-08,ISCO-88,English title
0,1,3117,n
1,1111,1110,Alderman
2,1111,1110,Alderwoman


In [15]:
# Find Column Header
title_col = next((c for c in ["English title", "english_title", "title_en", "Title"] if c in df_isco_idx.columns), None)
if title_col is None:
    raise ValueError(f"Keine Titelspalte gefunden. Spalten: {list(df_isco_idx.columns)}")

isco_choices_en_titles = (
    df_isco_idx[title_col]
    .astype("string").str.strip()
    .dropna()
)

# Filter out ‘n’ from row 2 in Excel
isco_choices_en_titles = isco_choices_en_titles[~isco_choices_en_titles.isin(["", "n", "N"])]
isco_choices_en_titles = isco_choices_en_titles.drop_duplicates().tolist()

print("ISCO choices (job titles):", len(isco_choices_en_titles))
isco_choices_en_titles[:20]

ISCO choices (job titles): 7010


['Alderman',
 'Alderwoman',
 'Chancellor, government',
 'Chief minister, government',
 'Chief whip',
 'Congressman',
 'Congresswoman',
 'Councillor, city',
 'Councillor, government',
 'Governor, Commonwealth',
 'Governor, State',
 'Governor-General',
 'Head of State',
 'Legislator',
 'Mayor',
 'Member, house of assembly',
 'Member, legilslative assembly',
 'Member, legilslative council',
 'Member, local government',
 'Member, parliament']

## 3. Sampling: DE 50 + EN 50

The DE sample is provided only as an example; 50 examples from German sources. The focus is on the EN sample and results: 50 examples from external English sources (random).

In [16]:
# Sampling: DE 50 + EN 50
RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)

SAMPLE_N_DE = 50
SAMPLE_N_EN = 50

# DE: half and half BA + Workwise
DE_SOURCES = {
    "ba_jobsuche_api_2025": 25,
    "workwise_selenium": 25,
}

# EN: mixed (without Udemy)
EN_SOURCES = {
    "kaggle_linkedin_2023_2024_big": 30,
    "kaggle_techsalerator": 10,
    "kaggle_resume_structured": 10,
}

def sample_from_sources(df, lang: str, plan: dict, target_n: int, seed=42):
    df_lang = df[df["language"].fillna("").str.lower().eq(lang)].copy()
    df_lang["job_title_raw"] = df_lang["job_title_raw"].astype(str).str.strip()
    df_lang = df_lang[df_lang["job_title_raw"].ne("")]

    df_lang = df_lang[df_lang["source_name"].isin(plan.keys())].copy()

    parts = []
    for src_name, n in plan.items():
        df_src = df_lang[df_lang["source_name"] == src_name]
        take = min(n, len(df_src))
        if take <= 0:
            print(f"Quelle leer: {src_name} ({lang})")
            continue
        parts.append(df_src.sample(n=take, random_state=seed))

    if not parts:
        return df_lang.head(0).copy()

    out = pd.concat(parts, ignore_index=True)

    # final exactly target_n
    if len(out) > target_n:
        out = out.sample(n=target_n, random_state=seed)

    return out.reset_index(drop=True)

In [17]:
# Output
df_de_sample = sample_from_sources(df_docs,lang="de",plan=DE_SOURCES,target_n=SAMPLE_N_DE,seed=RANDOM_SEED)
df_en_sample = sample_from_sources(df_docs,lang="en",plan=EN_SOURCES,target_n=SAMPLE_N_EN,seed=RANDOM_SEED)

print("DE sample:", df_de_sample.shape)
print(df_de_sample["source_name"].value_counts(dropna=False))

print("\nEN sample:", df_en_sample.shape)
print(df_en_sample["source_name"].value_counts(dropna=False))

display(df_de_sample[["doc_id","source_name","source_type","job_title_raw","language"]].head(3))
display(df_en_sample[["doc_id","source_name","source_type","job_title_raw","language"]].head(3))

DE sample: (50, 5)
source_name
ba_jobsuche_api_2025    25
workwise_selenium       25
Name: count, dtype: int64

EN sample: (50, 5)
source_name
kaggle_linkedin_2023_2024_big    30
kaggle_techsalerator             10
kaggle_resume_structured         10
Name: count, dtype: int64


,doc_id,source_name,source_type,job_title_raw,language
0,14751-373A62276-S,ba_jobsuche_api_2025,job_ad,Maschinenbediener (m/w/d),de
1,13635-542cccae_JB5001452-S,ba_jobsuche_api_2025,job_ad,MASSEUR / MED. BADEMEISTER (m/w/d) für unser T...,de
2,14751-373A56768-S,ba_jobsuche_api_2025,job_ad,Helfer - Bau (m/w/d),de


,doc_id,source_name,source_type,job_title_raw,language
0,3902944011,kaggle_linkedin_2023_2024_big,job_ad,Senior Automation Engineer - Power Systems,en
1,3901960222,kaggle_linkedin_2023_2024_big,job_ad,DISH Installation Technician - Field,en
2,3900944095,kaggle_linkedin_2023_2024_big,job_ad,Order Builder,en


## 4. Perform matching: DE->KldB, EN->ESCO, EN->KldB-EN-Proxy, EN->ISCO Index

Helper: Apply matching to a choice list

In [18]:
def run_matching(df_sample: pd.DataFrame, choices: list[str], target_name: str) -> pd.DataFrame: # Matches a sample's `job_title_raw` against `choices` and returns the best match and score
    rows = []
    for _, r in df_sample.iterrows():
        jt = r.get("job_title_raw", "")
        m, s = best_match(jt, choices)
        rows.append({
            "doc_id": r.get("doc_id"),
            "source_name": r.get("source_name"),
            "source_type": r.get("source_type"),
            "language": r.get("language"),
            "job_title_raw": jt,
            "match_target": target_name,
            "best_match": m if m is not None else "",
            "score": float(s) if s is not None else 0.0,
        })
    return pd.DataFrame(rows)

### 4.1 DE-Check: DE Job Titles -> KLDB Occupational Titles

In [19]:
df_match_de_kldb = run_matching(
    df_de_sample,
    kldb_choices_de,
    target_name="KldB_titles_DE"
)

df_match_de_kldb.head(10)

,doc_id,source_name,source_type,language,job_title_raw,match_target,best_match,score
0,14751-373A62276-S,ba_jobsuche_api_2025,job_ad,de,Maschinenbediener (m/w/d),KldB_titles_DE,Abfllmaschinenbediener/in,90.0
1,13635-542cccae_JB5001452-S,ba_jobsuche_api_2025,job_ad,de,MASSEUR / MED. BADEMEISTER (m/w/d) für unser T...,KldB_titles_DE,Agiler Coach / Scrum Master,85.5
2,14751-373A56768-S,ba_jobsuche_api_2025,job_ad,de,Helfer - Bau (m/w/d),KldB_titles_DE,Abteilungsleiter/in - Gewerbeaufsicht,85.5
3,14751-373A59701-S,ba_jobsuche_api_2025,job_ad,de,Mechaniker (m/w/d),KldB_titles_DE,Anlagenmechaniker/in,90.0
4,11858-15914135-STA-S,ba_jobsuche_api_2025,job_ad,de,Erzieher (m/w/d),KldB_titles_DE,Arbeitserzieher/in,90.0
5,14751-269A11662-S,ba_jobsuche_api_2025,job_ad,de,Kalkulator Bau (m/w/d),KldB_titles_DE,Baukalkulator/in,76.0
6,14751-373A61472-S,ba_jobsuche_api_2025,job_ad,de,Schlosser (m/w/d),KldB_titles_DE,Aufzugschlosser/in,90.0
7,12288-4658344373-S,ba_jobsuche_api_2025,job_ad,de,HR Assistant (m/w/d),KldB_titles_DE,Ditlehrassistent/in,75.0
8,12513-0007620327-S,ba_jobsuche_api_2025,job_ad,de,Erzieher (m/w/d),KldB_titles_DE,Arbeitserzieher/in,90.0
9,12288-4658311807-S,ba_jobsuche_api_2025,job_ad,de,Junior Assistenz (m/w/d),KldB_titles_DE,Fachkraft - Pflegeassistenz,64.8


### 4.2 EN Comparison: EN Job Titles -> ESCO Occupation Labels (EN, preferred+alt)

In [20]:
df_match_en_esco = run_matching(
    df_en_sample,
    esco_choices_en,
    target_name="ESCO_occ_labels_EN_pref+alt"
)

df_match_en_esco.head(10)

,doc_id,source_name,source_type,language,job_title_raw,match_target,best_match,score
0,3902944011,kaggle_linkedin_2023_2024_big,job_ad,en,Senior Automation Engineer - Power Systems,ESCO_occ_labels_EN_pref+alt,automation engineer,90.0
1,3901960222,kaggle_linkedin_2023_2024_big,job_ad,en,DISH Installation Technician - Field,ESCO_occ_labels_EN_pref+alt,metallurgical technician,85.5
2,3900944095,kaggle_linkedin_2023_2024_big,job_ad,en,Order Builder,ESCO_occ_labels_EN_pref+alt,warehouse order picker,85.5
3,3903878594,kaggle_linkedin_2023_2024_big,job_ad,en,"Mountain Multimedia Journalist, KMGH",ESCO_occ_labels_EN_pref+alt,journalist,90.0
4,3905670593,kaggle_linkedin_2023_2024_big,job_ad,en,Licensed Practical Nurse (LPN),ESCO_occ_labels_EN_pref+alt,specialist nurse,85.5
5,3901967275,kaggle_linkedin_2023_2024_big,job_ad,en,Human Resources Project Manager,ESCO_occ_labels_EN_pref+alt,human resources manager,95.0
6,3901373261,kaggle_linkedin_2023_2024_big,job_ad,en,Clinical Paramedic,ESCO_occ_labels_EN_pref+alt,paramedic in emergency responses,85.5
7,3903459065,kaggle_linkedin_2023_2024_big,job_ad,en,CCTV/Access Control Installer (regional travel),ESCO_occ_labels_EN_pref+alt,door installer,85.5
8,3902915545,kaggle_linkedin_2023_2024_big,job_ad,en,Java Architect,ESCO_occ_labels_EN_pref+alt,architect,90.0
9,3901947295,kaggle_linkedin_2023_2024_big,job_ad,en,Laboratory Operations (LabOps) Project Manager,ESCO_occ_labels_EN_pref+alt,project manager,90.0


### 4.3 EN Comparison: EN Job Titles -> KldB EN (automatically translated KldB job titles)

In [21]:
df_match_en_kldb_proxy = run_matching(
    df_en_sample,
    kldb_choices_en_translated,
    target_name="KldB_titles_EN_proxy_translated"
)

df_match_en_kldb_proxy.head(10)

,doc_id,source_name,source_type,language,job_title_raw,match_target,best_match,score
0,3902944011,kaggle_linkedin_2023_2024_big,job_ad,en,Senior Automation Engineer - Power Systems,KldB_titles_EN_proxy_translated,Waste management engineer,85.500000
1,3901960222,kaggle_linkedin_2023_2024_big,job_ad,en,DISH Installation Technician - Field,KldB_titles_EN_proxy_translated,Waste technician,85.500000
2,3900944095,kaggle_linkedin_2023_2024_big,job_ad,en,Order Builder,KldB_titles_EN_proxy_translated,Aggregate builder (electrical machines),85.500000
3,3903878594,kaggle_linkedin_2023_2024_big,job_ad,en,"Mountain Multimedia Journalist, KMGH",KldB_titles_EN_proxy_translated,Specialist - multimedia communication and pres...,85.500000
4,3905670593,kaggle_linkedin_2023_2024_big,job_ad,en,Licensed Practical Nurse (LPN),KldB_titles_EN_proxy_translated,Geriatric nurse,85.500000
5,3901967275,kaggle_linkedin_2023_2024_big,job_ad,en,Human Resources Project Manager,KldB_titles_EN_proxy_translated,Project manager,86.896552
6,3901373261,kaggle_linkedin_2023_2024_big,job_ad,en,Clinical Paramedic,KldB_titles_EN_proxy_translated,Assistant - clinical studies,85.500000
7,3903459065,kaggle_linkedin_2023_2024_big,job_ad,en,CCTV/Access Control Installer (regional travel),KldB_titles_EN_proxy_translated,Antenna installer,85.500000
8,3902915545,kaggle_linkedin_2023_2024_big,job_ad,en,Java Architect,KldB_titles_EN_proxy_translated,Garden and landscape architect,85.500000
9,3901947295,kaggle_linkedin_2023_2024_big,job_ad,en,Laboratory Operations (LabOps) Project Manager,KldB_titles_EN_proxy_translated,Project manager,86.896552


### 4.4 EN Comparison: EN Job Titles -> ISCO Index Job Titles (EN) (ISCO08_Index.xlsx)

In [22]:
df_match_en_isco = run_matching(
    df_en_sample,
    isco_choices_en_titles,
    target_name="ISCO08_Index_jobtitles_EN"
)

df_match_en_isco.head(10)

,doc_id,source_name,source_type,language,job_title_raw,match_target,best_match,score
0,3902944011,kaggle_linkedin_2023_2024_big,job_ad,en,Senior Automation Engineer - Power Systems,ISCO08_Index_jobtitles_EN,"Director, power station",85.5
1,3901960222,kaggle_linkedin_2023_2024_big,job_ad,en,DISH Installation Technician - Field,ISCO08_Index_jobtitles_EN,"Technician, field crop",85.5
2,3900944095,kaggle_linkedin_2023_2024_big,job_ad,en,Order Builder,ISCO08_Index_jobtitles_EN,"Labourer, builder's",75.0
3,3903878594,kaggle_linkedin_2023_2024_big,job_ad,en,"Mountain Multimedia Journalist, KMGH",ISCO08_Index_jobtitles_EN,"Author, multimedia",85.5
4,3905670593,kaggle_linkedin_2023_2024_big,job_ad,en,Licensed Practical Nurse (LPN),ISCO08_Index_jobtitles_EN,"Anaesthetist, nurse",85.5
5,3901967275,kaggle_linkedin_2023_2024_big,job_ad,en,Human Resources Project Manager,ISCO08_Index_jobtitles_EN,"Builder, project",85.5
6,3901373261,kaggle_linkedin_2023_2024_big,job_ad,en,Clinical Paramedic,ISCO08_Index_jobtitles_EN,"Manager, project: clinical trials",85.5
7,3903459065,kaggle_linkedin_2023_2024_big,job_ad,en,CCTV/Access Control Installer (regional travel),ISCO08_Index_jobtitles_EN,"Engineer, air pollution control",85.5
8,3902915545,kaggle_linkedin_2023_2024_big,job_ad,en,Java Architect,ISCO08_Index_jobtitles_EN,"Architect, naval",76.0
9,3901947295,kaggle_linkedin_2023_2024_big,job_ad,en,Laboratory Operations (LabOps) Project Manager,ISCO08_Index_jobtitles_EN,"Builder, project",85.5


## 5. Evaluation: Summary of Metrics + Worst/Best Matches

### 5.1 Summary Table

Summary Function:

In [23]:
def summarize_scores(df: pd.DataFrame, label: str) -> pd.Series:
    s = df["score"].astype(float)
    return pd.Series({
        "label": label,
        "n": len(df),
        "mean": s.mean(),
        "median": s.median(),
        "p>=95": (s >= 95).mean(),
        "p>=90": (s >= 90).mean(),
        "p>=85": (s >= 85).mean(),
        "p>=80": (s >= 80).mean(),
        "p>=70": (s >= 70).mean(),
    })

Summary Table (DE + EN):

In [24]:
summary = pd.DataFrame([
    summarize_scores(df_match_de_kldb, "DE -> KldB(DE)"),
    summarize_scores(df_match_en_esco, "EN -> ESCO(pref+alt)"),
    summarize_scores(df_match_en_kldb_proxy, "EN -> KldB EN Proxy"),
    summarize_scores(df_match_en_isco, "EN -> ISCO Index Titles"),
])

# Format Values
pct_cols = [c for c in summary.columns if c.startswith("p>=")]
summary[pct_cols] = (summary[pct_cols] * 100).round(1)
summary

,label,n,mean,median,p>=95,p>=90,p>=85,p>=80,p>=70
0,DE -> KldB(DE),50,81.562754,85.5,2.0,22.0,62.0,64.0,88.0
1,EN -> ESCO(pref+alt),50,86.545653,85.5,8.0,26.0,96.0,96.0,96.0
2,EN -> KldB EN Proxy,50,86.056924,85.5,4.0,14.0,94.0,98.0,100.0
3,EN -> ISCO Index Titles,50,81.256768,85.5,0.0,2.0,66.0,70.0,92.0


**Interpretation of the Summary Results:**

The summary table shows clear differences between the matching targets tested:
- The DE → KldB matching serves as a plausibility check and, as expected, achieves very high scores. German job titles map well to German KldB occupational designations.
- For English job titles, matching against ESCO Occs yields the best results across all metrics.
- Matching against translated KldB occupational designations is only slightly below ESCO. The differences are small, particularly in the median and upper quantiles.
- Matching against ISCO job titles performs significantly worse. This confirms that ISCO index titles are only of limited suitability for job title matching.

Overall, the results suggest that ESCO Occupations, closely followed by EN-KLDB occupational designations, represent the most robust target for English-language job titles.

### 5.2 Worst/Best per Matching (detailed output)

In [25]:
def show_examples(df: pd.DataFrame, n=10):
    df2 = df.copy()
    df2["job_title_raw_norm"] = df2["job_title_raw"].apply(normalize_title)
    df2["best_match_norm"] = df2["best_match"].apply(normalize_title)

    print("BEST")
    display(df2.sort_values("score", ascending=False).head(n)[
        ["source_name","job_title_raw","best_match","score"]
    ])

    print("WORST")
    display(df2.sort_values("score", ascending=True).head(n)[
        ["source_name","job_title_raw","best_match","score"]
    ])

show_examples(df_match_de_kldb, n=10)
show_examples(df_match_en_esco, n=10)
show_examples(df_match_en_kldb_proxy, n=10)
show_examples(df_match_en_isco, n=10)

BEST


,source_name,job_title_raw,best_match,score
14,ba_jobsuche_api_2025,Krankenschwester/-pfleger (m/w/d),Krankenschwester/-pfleger,96.0
0,ba_jobsuche_api_2025,Maschinenbediener (m/w/d),Abfllmaschinenbediener/in,90.0
11,ba_jobsuche_api_2025,Assistenz (m/w/d),Fachkraft - Pflegeassistenz,90.0
3,ba_jobsuche_api_2025,Mechaniker (m/w/d),Anlagenmechaniker/in,90.0
8,ba_jobsuche_api_2025,Erzieher (m/w/d),Arbeitserzieher/in,90.0
4,ba_jobsuche_api_2025,Erzieher (m/w/d),Arbeitserzieher/in,90.0
6,ba_jobsuche_api_2025,Schlosser (m/w/d),Aufzugschlosser/in,90.0
24,ba_jobsuche_api_2025,Elektriker (m/w/d),Anlagenelektriker/in (Elektroanlageninstallation),90.0
21,ba_jobsuche_api_2025,Schlosser (m/w/d),Aufzugschlosser/in,90.0
19,ba_jobsuche_api_2025,Dachdecker (m/w/d),Flachdachdecker/in,90.0


WORST


,source_name,job_title_raw,best_match,score
49,workwise_selenium,Logopäde (m/w/d) für unseren Standort in Betzdorf,Kantor/in,60.000000
23,ba_jobsuche_api_2025,Jungbauleiter SF-Bau (w/m/d),Bauleiter/in (Ausbau),63.414634
9,ba_jobsuche_api_2025,Junior Assistenz (m/w/d),Fachkraft - Pflegeassistenz,64.800000
41,workwise_selenium,Fachplaner für Gebäudeversorgungstechnik HKLS ...,Gasversorgungstechniker/in,66.122449
33,workwise_selenium,Monteur für Sicherheitssysteme (m/w/d),Remonteur/in,66.315789
25,workwise_selenium,Techniker Sanitärhandwerk (w/m/d),Meister/in - Chirurgiemechanikerhandwerk,68.571429
27,workwise_selenium,Maschinenführer für die Zargenfertigung (m/w/d),Baumaschinenfhrer/in,70.000000
28,workwise_selenium,Senior Bauzeichner für CAD Zeichnungen (m/w/d),Tenor,72.000000
47,workwise_selenium,Senior Consultant für SAP Security (m/w/d),Tenor,72.000000
15,ba_jobsuche_api_2025,Lagerhelfer (m/w/d) 2-Schicht,Lagerhelfer/in,72.000000


BEST


...

,source_name,job_title_raw,best_match,score
41,kaggle_resume_structured,Cyber Surety Journeyman,journeyman potter,70.37037
15,kaggle_linkedin_2023_2024_big,Part-Time Bookkeeper (10 hours/week),Zookeeper,80.00000
16,kaggle_linkedin_2023_2024_big,Airport Ramp Agent- F9,Ramp agent,81.00000
2,kaggle_linkedin_2023_2024_big,Order Builder,Aggregate builder (electrical machines),85.50000
4,kaggle_linkedin_2023_2024_big,Licensed Practical Nurse (LPN),Geriatric nurse,85.50000
6,kaggle_linkedin_2023_2024_big,Clinical Paramedic,Assistant - clinical studies,85.50000
7,kaggle_linkedin_2023_2024_big,CCTV/Access Control Installer (regional travel),Antenna installer,85.50000
0,kaggle_linkedin_2023_2024_big,Senior Automation Engineer - Power Systems,Waste management engineer,85.50000
13,kaggle_linkedin_2023_2024_big,Radiologic Technologist,Mining technologist - civil engineering techno...,85.50000
8,kaggle_linkedin_2023_2024_big,Java Architect,Garden and landscape architect,85.50000


BEST


,source_name,job_title_raw,best_match,score
24,kaggle_linkedin_2023_2024_big,Technical Support,"Agent, technical support: information technology",90.000000
47,kaggle_resume_structured,Database Administrator,"Administrator, database",88.666667
3,kaggle_linkedin_2023_2024_big,"Mountain Multimedia Journalist, KMGH","Author, multimedia",85.500000
4,kaggle_linkedin_2023_2024_big,Licensed Practical Nurse (LPN),"Anaesthetist, nurse",85.500000
5,kaggle_linkedin_2023_2024_big,Human Resources Project Manager,"Builder, project",85.500000
6,kaggle_linkedin_2023_2024_big,Clinical Paramedic,"Manager, project: clinical trials",85.500000
9,kaggle_linkedin_2023_2024_big,Laboratory Operations (LabOps) Project Manager,"Builder, project",85.500000
7,kaggle_linkedin_2023_2024_big,CCTV/Access Control Installer (regional travel),"Engineer, air pollution control",85.500000
11,kaggle_linkedin_2023_2024_big,"Research Intern, Organizational Research and C...","Director, policy and planning",85.500000
10,kaggle_linkedin_2023_2024_big,Commissioned Sales Associate (Palm Beach),"Director, sales",85.500000


WORST


,source_name,job_title_raw,best_match,score
18,kaggle_linkedin_2023_2024_big,"VP, Strategy & Insights","Analyst, strategy",61.750000
30,kaggle_techsalerator,Frontend Developer (f/m/div.),"Developer, print",64.125000
16,kaggle_linkedin_2023_2024_big,Airport Ramp Agent- F9,"Attendant, airport: ramp",66.086957
39,kaggle_techsalerator,Material Handler Forklift,"Crater, hand",67.500000
41,kaggle_resume_structured,Cyber Surety Journeyman,Nurseryman,70.000000
45,kaggle_resume_structured,Sr. UX Designer / UI Developer,Cooper,72.000000
36,kaggle_techsalerator,DevOps Engineer,Miner,72.000000
48,kaggle_resume_structured,Teacher/Tutor,Author,72.000000
49,kaggle_resume_structured,Sr. Android Application Developer,"Manager, application development",73.846154
2,kaggle_linkedin_2023_2024_big,Order Builder,"Labourer, builder's",75.000000


### 5.3 Comparison on the same EN-50: Which one performs best per document?

Which target achieved the highest score (ESCO vs. KldB-Proxy vs. ISCO) per job title:

In [26]:
# Merge EN by doc_id
en_compare = (
    df_match_en_esco[["doc_id","source_name","job_title_raw","best_match","score"]]
      .rename(columns={"best_match":"best_esco", "score":"score_esco"})
    .merge(
        df_match_en_kldb_proxy[["doc_id","best_match","score"]]
          .rename(columns={"best_match":"best_kldb_proxy", "score":"score_kldb_proxy"}),
        on="doc_id",
        how="left"
    )
    .merge(
        df_match_en_isco[["doc_id","best_match","score"]]
          .rename(columns={"best_match":"best_isco", "score":"score_isco"}),
        on="doc_id",
        how="left"
    )
)

# Winners per line
score_cols = ["score_esco","score_kldb_proxy","score_isco"]
en_compare["winner"] = en_compare[score_cols].idxmax(axis=1).str.replace("score_", "")
en_compare["winner_score"] = en_compare[score_cols].max(axis=1)

en_compare.head(10)

,doc_id,source_name,job_title_raw,best_esco,score_esco,best_kldb_proxy,score_kldb_proxy,best_isco,score_isco,winner,winner_score
0,3902944011,kaggle_linkedin_2023_2024_big,Senior Automation Engineer - Power Systems,automation engineer,90.0,Waste management engineer,85.500000,"Director, power station",85.5,esco,90.0
1,3901960222,kaggle_linkedin_2023_2024_big,DISH Installation Technician - Field,metallurgical technician,85.5,Waste technician,85.500000,"Technician, field crop",85.5,esco,85.5
2,3900944095,kaggle_linkedin_2023_2024_big,Order Builder,warehouse order picker,85.5,Aggregate builder (electrical machines),85.500000,"Labourer, builder's",75.0,esco,85.5
3,3903878594,kaggle_linkedin_2023_2024_big,"Mountain Multimedia Journalist, KMGH",journalist,90.0,Specialist - multimedia communication and pres...,85.500000,"Author, multimedia",85.5,esco,90.0
4,3905670593,kaggle_linkedin_2023_2024_big,Licensed Practical Nurse (LPN),specialist nurse,85.5,Geriatric nurse,85.500000,"Anaesthetist, nurse",85.5,esco,85.5
5,3901967275,kaggle_linkedin_2023_2024_big,Human Resources Project Manager,human resources manager,95.0,Project manager,86.896552,"Builder, project",85.5,esco,95.0
6,3901373261,kaggle_linkedin_2023_2024_big,Clinical Paramedic,paramedic in emergency responses,85.5,Assistant - clinical studies,85.500000,"Manager, project: clinical trials",85.5,esco,85.5
7,3903459065,kaggle_linkedin_2023_2024_big,CCTV/Access Control Installer (regional travel),door installer,85.5,Antenna installer,85.500000,"Engineer, air pollution control",85.5,esco,85.5
8,3902915545,kaggle_linkedin_2023_2024_big,Java Architect,architect,90.0,Garden and landscape architect,85.500000,"Architect, naval",76.0,esco,90.0
9,3901947295,kaggle_linkedin_2023_2024_big,Laboratory Operations (LabOps) Project Manager,project manager,90.0,Project manager,86.896552,"Builder, project",85.5,esco,90.0


In [27]:
# Absolute numbers
winner_counts = en_compare["winner"].value_counts()
winner_counts

winner
esco          41
kldb_proxy     8
isco           1
Name: count, dtype: int64

In terms of whole numbers, the EN comparison shows that matching via ESCO most often yields the greatest alignment. The Winner comparison confirms the results of the aggregated metrics.

## 6. Export

In [28]:
OUT_XLSX = DATA_PROCESSED_EXTERNAL / "jobtitle_qualitycheck_09.xlsx"

with pd.ExcelWriter(OUT_XLSX, engine="openpyxl") as writer:
    summary.to_excel(writer, sheet_name="summary", index=False)

    df_match_de_kldb.sort_values("score", ascending=False).to_excel(writer, sheet_name="de_kldb", index=False)
    df_match_en_esco.sort_values("score", ascending=False).to_excel(writer, sheet_name="en_esco", index=False)
    df_match_en_kldb_proxy.sort_values("score", ascending=False).to_excel(writer, sheet_name="en_kldb_proxy", index=False)
    df_match_en_isco.sort_values("score", ascending=False).to_excel(writer, sheet_name="en_isco", index=False)

    en_compare.sort_values("winner_score", ascending=False).to_excel(writer, sheet_name="en_compare", index=False)

print("Saved:", OUT_XLSX)

Saved: C:\Users\sigle\OneDrive - Hochschule Reutlingen\Dokumente\Profilerweiterung\data\processed_external\jobtitle_qualitycheck_09.xlsx


# Conclusion: Notebook 09

The goal of this notebook was to create a methodologically sound decision-making tool for job title matching as part of the profile expansion. To this end, various standard target systems were systematically compared.

Results:
- Matching job titles to KldB occupational designations works consistently and serves as a reliable starting point.
- For external English data, matching against ESCO Occupations yields the best results: the highest average and median scores, as well as the largest proportion of very good matches.
- Matching against translated KldB occupational designations is a very good alternative. The scores are only slightly lower than those of ESCO, making this approach particularly attractive given the desire for a stronger link to the KldB classification system. Additionally, this approach may be simpler and less prone to errors compared to ESCO matching.
- Matching against ISCO performs worse and will not be considered further for the profile expansion.

Decision on the next steps:
- For English-language external job titles, it would primarily make sense to rely on ESCO matching, as this achieves the highest degree of alignment and fits well conceptually with the skill-based profile expansion.
- Alternatively, KldB EN matching will be used, as it yields nearly comparable results and enables direct integration with existing KldB-SOLL profiles. Additionally, this allows for a simpler methodological approach, which is why this matching method will be applied moving forward.

## Additionally, create a new file containing KLDB-EN-Proxy job title translations using kldb_5_code

Useful for further steps in the profile expansion: German KldB titles contain the kldb_5_code (in kldb_job_titles_long.parquet). The proxy file kldb_titles_en_proxy.parquet contains: kldb_title_de & kldb_title_en_proxy; this allows the KLDB 5 codes to be added via a join (join logic: left join on kldb_title_de == job_title_de). Save to a new file. Direct KLDB-5 code is required for later job title matching.

In [29]:
import pandas as pd

# load proxy file (de -> en proxy)
proxy_path = DATA_PROCESSED_EXTERNAL / "kldb_titles_en_proxy.parquet"
df_proxy = pd.read_parquet(proxy_path)

# Columns: kldb_title_de, kldb_title_en_proxy
df_proxy["kldb_title_de"] = df_proxy["kldb_title_de"].astype(str).str.strip()

# load KldB job titles long with codes
kldb_long_path = KDB_JOB_TITLES_LONG_PATH
df_kldb_long = pd.read_parquet(kldb_long_path)

# find columns 
candidate_title_cols = ["job_title_de", "kldb_job_title_de", "berufsbenennung", "job_title", "title_de", "kldb_title_de"]
candidate_code_cols  = ["kldb_5_code", "kldb_code", "kldb5", "code_5", "kldb_code_5"]

col_title = next((c for c in candidate_title_cols if c in df_kldb_long.columns), None)
col_code  = next((c for c in candidate_code_cols  if c in df_kldb_long.columns), None)

if col_title is None or col_code is None:
    raise ValueError(
        f"Could not find title/code columns in df_kldb_long. "
        f"Columns present: {list(df_kldb_long.columns)}"
    )

df_kldb_map = df_kldb_long[[col_title, col_code]].copy()
df_kldb_map[col_title] = df_kldb_map[col_title].astype(str).str.strip()

# Remove Duplicates
df_kldb_map = df_kldb_map.dropna(subset=[col_title, col_code]).drop_duplicates()

# sanity check: DE Titles with multiple codes happens
multi = (df_kldb_map.groupby(col_title)[col_code].nunique().sort_values(ascending=False))
n_multi = int((multi > 1).sum())
if n_multi > 0:
    print(f"{n_multi} DE titles map to multiple different codes. "
          f"Example titles:\n{multi[multi > 1].head(10)}")

# Get proxy title kldb_5_code via DE title
df_proxy_with_code = df_proxy.merge(
    df_kldb_map.rename(columns={col_title: "kldb_title_de", col_code: "kldb_5_code"}),
    on="kldb_title_de",
    how="left",
)

# Quality
missing_codes = df_proxy_with_code["kldb_5_code"].isna().sum()
print("Proxy rows:", len(df_proxy_with_code))
print("Missing kldb_5_code after merge:", missing_codes)

# Save as a new document, CSV, and Parquet
out_path = DATA_PROCESSED_EXTERNAL / "kldb_titles_en_proxy_with_code.parquet"
df_proxy_with_code.to_parquet(out_path, index=False)
print("Saved:", out_path)
out_csv = DATA_PROCESSED_EXTERNAL / "kldb_titles_en_proxy_with_code.csv"
df_proxy_with_code.to_csv(out_csv, index=False, encoding="utf-8")
print("Saved:", out_csv)

1 DE titles map to multiple different codes. Example titles:
job_title_de
Dipl.-Ing. - Informationstechnik    2
Name: kldb_5_code, dtype: int64
Proxy rows: 18838
Missing kldb_5_code after merge: 0
Saved: C:\Users\sigle\OneDrive - Hochschule Reutlingen\Dokumente\Profilerweiterung\data\processed_external\kldb_titles_en_proxy_with_code.parquet
Saved: C:\Users\sigle\OneDrive - Hochschule Reutlingen\Dokumente\Profilerweiterung\data\processed_external\kldb_titles_en_proxy_with_code.csv
